# Figure 6e — 5-FU's morphological neighbourhood

Does 5-FU's morphological neighbourhood survive loss of p53?

Compound set is DATA-DEFINED: 5-FU + its top-10 nearest neighbours in HCT116
(read from fig_5fu_top10_neighbours_HCT116.csv).

Doses are FROZEN FROM HCT116: each compound's concentration = the one most similar
to 5-FU in HCT116; those same doses are read out in HT29 (nearest available dose if
not run in HT29). This avoids the "max over concentrations" artifact that fishes up
spurious similarity for weakly-responding compounds in p53-mutant HT29.

Features: each cell line uses ITS OWN feature set (the parquets carry per-line
feature selection). The dissolution is a within-line property, so a common set is
not needed -- and using HCT116's own features reproduces the published 5-FU
neighbour similarities.

Grit fading: per line, any compound whose grit (at its shown dose) does NOT pass
1.96 is rendered SEE-THROUGH (its row/column faded) -- so non-reproducible
compounds (e.g. the MDM2 inhibitors in HT29) visually recede.

    python fig_5fu_neighbours_frozen_doses.py


In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import profiles
from utils.panels import save_panel


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# keep SVG text as editable <text> (not outlined paths), in Arial
plt.rcParams.update({"svg.fonttype": "none", "font.family": "Arial",
                     "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"]})

RES    = profiles("exp1_main", "")
ANCHOR = "Fluor"
GRIT   = 1.96

load = lambda cl: pd.read_parquet(RES / f"grit_data_aggregates_{cl}.parquet").dropna(axis=1, how="all")
cos  = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
feats_of = lambda d: [c for c in d.columns if not c.startswith("Metadata_") and d[c].dtype.kind in "fi"]

H, T = load("HCT116"), load("HT29")
FCH, FCT = feats_of(H), feats_of(T)          # each line keeps its OWN feature set

# 5-FU + its top-10 HCT116 neighbours, but Vinorelbine dropped (its 5-FU-matched dose
# fails grit in HCT116) and replaced by the next-ranked neighbour, Crizotinib (#11).
ORDER = ["Fluor", "AMG23", "Nutli", "SN-38", "abema", "Trifl", "Gemci", "Pacli", "BMS-7", "Borte", "Crizo"]
LAB   = ["5-FU", "AMG-232", "Nutlin-3", "SN-38", "Abemaciclib", "Trifluridine",
         "Gemcitabine", "Paclitaxel", "BMS-754807", "Bortezomib", "Crizotinib"]
n = len(ORDER)

def prof(d, code, conc, F):
    s = d[(d.Metadata_pert_type == "trt") & (d.Metadata_name == code) & (d.Metadata_cmpd_conc == conc)]
    return s[F].mean().values if len(s) else None
def concs(d, code):
    return sorted(d[(d.Metadata_pert_type == "trt") & (d.Metadata_name == code)].Metadata_cmpd_conc.unique())
def grit_at(d, code, conc):
    s = d[(d.Metadata_pert_type == "trt") & (d.Metadata_name == code) & (d.Metadata_cmpd_conc == conc)]
    return float(s["Metadata_grit"].median()) if len(s) else np.nan

# freeze HCT116 doses (each compound's dose most similar to 5-FU, in HCT116 features)
fu_dose = (H[(H.Metadata_pert_type == "trt") & (H.Metadata_name == ANCHOR)]
           .groupby("Metadata_cmpd_conc")["Metadata_grit"].mean().idxmax())
fu_ref  = prof(H, ANCHOR, fu_dose, FCH)
dose = {ANCHOR: fu_dose}
for c in ORDER[1:]:
    ok = [k for k in concs(H, c) if grit_at(H, c, k) >= GRIT]   # only grit-passing HCT116 doses
    dose[c] = max(ok or concs(H, c), key=lambda k: cos(prof(H, c, k, FCH), fu_ref))

def readout(d, F):
    P, used = {}, {}
    for c in ORDER:
        dc = dose[c] if prof(d, c, dose[c], F) is not None else min(concs(d, c), key=lambda x: abs(x - dose[c]))
        P[c], used[c] = prof(d, c, dc, F), dc
    M = np.array([[cos(P[a], P[b]) for b in ORDER] for a in ORDER])
    passes = np.array([grit_at(d, c, used[c]) >= GRIT for c in ORDER])
    return M, passes
MH, passH = readout(H, FCH)
MT, passT = readout(T, FCT)

vmin = float(np.floor(min(MH.min(), MT.min()) * 10) / 10)      # auto lower bound (no fixed -1)
vmax = 1.0

fig, axes = plt.subplots(1, 2, figsize=(13, 6.4))
for ax, M, passes, title in [(axes[0], MH, passH, "HCT116 (p53-WT)"),
                             (axes[1], MT, passT, "HT29 (p53-mutant) — same frozen doses")]:
    A = np.where(np.outer(passes, passes), 1.0, 0.18)          # fade if either compound fails grit
    ax.imshow(M, cmap="RdBu_r", vmin=vmin, vmax=vmax, alpha=A)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(LAB, rotation=45, ha="right", fontsize=7.5); ax.set_yticklabels(LAB, fontsize=7.5)
    for t, p in zip(ax.get_xticklabels(), passes):
        if not p: t.set_color("#bbbbbb")
    for t, p in zip(ax.get_yticklabels(), passes):
        if not p: t.set_color("#bbbbbb")
    for i in range(n):
        for j in range(n):
            a = 1.0 if (passes[i] and passes[j]) else 0.35
            ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=5.5,
                    color=("white" if abs(M[i,j]) > 0.55 else "#333"), alpha=a)
    ax.add_patch(plt.Rectangle((-0.5, -0.5), 1, n, fill=False, edgecolor="#993556", lw=1.3))
    ax.add_patch(plt.Rectangle((-0.5, -0.5), n, 1, fill=False, edgecolor="#993556", lw=1.3))
    ax.set_title(title, fontsize=10, pad=8)
cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap="RdBu_r"),
                  ax=axes, shrink=0.72, pad=0.02)
cb.set_label("cosine similarity (per-line features; doses frozen from HCT116 5-FU-match)")
fig.suptitle("5-FU and its top-10 HCT116 neighbours — frozen doses; compounds failing grit>1.96 are faded",
             y=1.02, fontsize=11)
save_panel(fig, "Fig6e",
           data=pd.concat([pd.DataFrame(MH, index=LAB, columns=LAB).round(3).assign(cell_line="HCT116"),
                           pd.DataFrame(MT, index=LAB, columns=LAB).round(3).assign(cell_line="HT29")]),
           caption="5-FU and its top-10 HCT116 neighbours at frozen doses, HCT116 vs HT29",
           notebook="analysis/3_Figure6/3_Fig6e_5fu_neighbours.ipynb")

print(f"colour range: [{vmin}, {vmax}]")
print("HT29 grit fail (faded):", [LAB[i] for i in range(n) if not passT[i]])
print("\nneighbour similarity TO 5-FU, HCT116 -> HT29:")
for i in range(1, n):
    print(f"   {LAB[i]:13s} {MH[0,i]:+.2f} -> {MT[0,i]:+.2f}   {'(HT29 grit fail)' if not passT[i] else ''}")
